# Personalized Product Recommender

Objective:
- To recommend relevant and similar products by leveraging product attributes and recent user interaction context, enabling personalized product discovery and seamless navigation across large product catalogs.

Business Use Cases
- Session-based product recommendation: Recommend the next best products based on items recently viewed by a user within a browsing session.

ML Framing:
- Type: Unsupervised Learning (Content-Based Recommendation)
- Target Output: Ranked List of recommended products based on similarity to recent user interations

### Architecture Overview

1. PDF ingestion and chunking
2. Semantic embedding using OpenAI embeddings
3. Vector storage using ChromaDB with persistence
4. Retrieval of top-k relevant chunks
5. Answer generation using an LLM constrained by retrieved context

### Preprocessing

1. Import libraries
2. Load OpenAI API key
3. Set and create Spark session
4. Load dataset

1. Import libraries

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans
import plotly.express as px

os.chdir(r'C:\Users\ashle\Project\usecase')

2. Load OpenAI API Key

In [2]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var. Set it and restart kernel."

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

3. Set and create Spark session

In [3]:
# Set Spark to use same Python as venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .master("local[*]")
         .appName("ProductRecommender")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.ui.enabled", "false")
         .getOrCreate())



In [4]:
spark = SparkSession.builder.appName("ProductRecommender").getOrCreate()
spark

4. Load dataset

In [5]:
file_path = 'data/products_dataset.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True, samplingRatio=1)
print(df.count())
df.show()

2000
+----------+--------------------+--------------------+
|product_id|               title|         description|
+----------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|
|        P1|Turmode 30 ft. RP...|If you need more ...|
|        P2|Large Tapestry Bo...|Polyester cover r...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|
|        P9|Traditional Silve...|This transitional...|
|       P10|15 in. x 59 in. O...|Its easy to add a...|
|       P11|1 qt. #350F-7 Wil...|BEHR PREMIUM PLUS...|
|       P12|Anthracite Cordle...|BlindsAvenue ligh...|
|       P13|SlimGrip 78-Inch ...|Luverne SlimGrip ...|
|       P14|6 in. x 28 in. x ...|Our Rustic Collec...|
|    

### Feature Engineering

 1. Text combination
 2. Embedding generation
 3. Feature assembly

 1. Text combination

 - Combine title and description column

In [6]:
df = df.withColumn('combined_text', concat_ws(" ", df.title,df.description))
df.show()

+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|       combined_text|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|
|        P9|Traditional Silve...|This transitional...|Traditional Silve...|
|       P10|

- Convert the combined_text into a list for embedding

In [7]:
list_combined_text = df.select('combined_text').rdd.flatMap(lambda x: x).collect()
print(list_combined_text[:2])

["Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.", "Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range."]


 2. Embedding generation

 - Use OpenAI text embedding model to create the vector embeddings

In [8]:
response = client.embeddings.create(
    input=list_combined_text,
    model="text-embedding-3-small",
    dimensions=512
)

- Display first two embedding vectors

In [9]:
embedding_vectors = [data.embedding for data in response.data]
embedding_vectors[:2]  # show first 2 embedding vectors

[[0.04267967492341995,
  0.02093353308737278,
  -0.013637710362672806,
  -0.002072375500574708,
  0.0031974425073713064,
  -0.03727405145764351,
  0.027204759418964386,
  0.07829317450523376,
  0.054974813014268875,
  -0.0628182590007782,
  0.0441989004611969,
  0.04829728230834007,
  -0.0678352415561676,
  0.025226231664419174,
  0.022541087120771408,
  0.0639488473534584,
  0.10104624927043915,
  -0.030843837186694145,
  -0.0889630988240242,
  0.07066170871257782,
  -0.05801326408982277,
  0.06719928979873657,
  -0.034818559885025024,
  -0.07377082854509354,
  0.03450058028101921,
  0.04328029975295067,
  -0.0525369830429554,
  0.025014245882630348,
  0.05112374946475029,
  -0.017250290140509605,
  -0.0031400297302752733,
  -0.022081784904003143,
  0.019131658598780632,
  -0.02457261085510254,
  0.04497617855668068,
  -0.06666932255029678,
  -0.021110186353325844,
  0.10450867563486099,
  -0.038687288761138916,
  0.02563253603875637,
  0.02345968782901764,
  -0.052748966962099075,
  

- Convert embedding vectors list into a Pyspark DataFrame

In [10]:
features_column_names = [f"embedding_{i}" for i in range(len(embedding_vectors[0]))]
embedding_df = spark.createDataFrame(embedding_vectors, schema=features_column_names)
print(embedding_df.count())
embedding_df.show()

2000
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------

Adding unique row_id to each row in embedding_df

In [11]:
embedding_df= embedding_df.repartition(1).withColumn("id", F.monotonically_increasing_id())
embedding_df.show(2)

+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+

Adding unique row_id to each row in main df

In [12]:
df = df.repartition(1).withColumn("id", F.monotonically_increasing_id())
df.show(2)

+----------+--------------------+--------------------+--------------------+---+
|product_id|               title|         description|       combined_text| id|
+----------+--------------------+--------------------+--------------------+---+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|  0|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|  1|
+----------+--------------------+--------------------+--------------------+---+
only showing top 2 rows


Join embedding_df into main df

In [13]:
df = df.join(embedding_df, on="id", how="inner").drop("id")
df.show(2)

+----------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------

In [14]:
df.count()

2000

 3. Feature assembly

 - Assemble the 512 embedding columns into a single 'embeddings' column
 - VectorAssembler is used to combine features that are going to be used in the machine learning model

In [15]:
assembler = VectorAssembler(
    inputCols=features_column_names,
    outputCol="embeddings"
)
data = assembler.transform(df)
data = data.select("product_id", "title", "description", "embeddings")
print(data.count())
data.show(2)

2000
+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|          embeddings|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04267967492341...|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|
+----------+--------------------+--------------------+--------------------+
only showing top 2 rows


### Modeling

1. Clustering model
2. Dimensionality Reduction
3. Visualization
4. Recommendation Engine
5. Demo

1. Clustering Model

- Apply K-means clustering with 5 clusters on the embeddings column
- K-means: Unsupervised machine learning model, used to cluster the data points based on features

In [16]:
kmeans = KMeans(k=5, featuresCol='embeddings', predictionCol='cluster')
model = kmeans.fit(data)
clustered_data = model.transform(data)
print(clustered_data.count())
clustered_data.show(5)

2000
+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|          embeddings|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04267967492341...|      4|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|
+----------+--------------------+--------------------+--------------------+-------+
only showing top 5 rows


2. Dimensionality Reduction

- To prepare for visualization, reduce the dimensionality of the embeddings from 512 dimensionas into 2 dimensions
- Principal Component Analysis (PCA): A dimensionality reduction technique to find the most important components of the data, and represent the high dimensional data in a lower dimensional space while retaining as much information possible.

In [17]:
pca = PCA(k=2, inputCol='embeddings', outputCol='pca_embeddings')
pca_model = pca.fit(clustered_data)
pca_results = pca_model.transform(clustered_data)
pca_results.show(5)

+----------+--------------------+--------------------+--------------------+-------+--------------------+
|product_id|               title|         description|          embeddings|cluster|      pca_embeddings|
+----------+--------------------+--------------------+--------------------+-------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04267967492341...|      4|[0.18867785148774...|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|[-0.1739559627059...|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|[-0.0202184231370...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|[0.00737630557938...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|[-0.0240373666866...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|[0.06058844178915...|      1|[-0.0014908469519...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|[

Convert to pandas dataframe

In [18]:
pca_df = pca_results.select("product_id", "pca_embeddings", "cluster").toPandas()
pca_df

,product_id,pca_embeddings,cluster
0,P0,"[0.18867785148774485, 0.034228364365063484]",4
1,P1,"[-0.17395596270599445, -0.13300941332495123]",4
2,P2,"[-0.020218423137017858, 0.31632337758792767]",2
3,P3,"[0.007376305579386828, 0.0593310913334424]",0
4,P4,"[-0.024037366686663137, -0.04418865487149011]",4
...,...,...,...
1995,P1995,"[0.1509553312518968, 0.29626086048325495]",1
1996,P1996,"[-0.056687769388258105, 0.5848000714343616]",2
1997,P1997,"[0.10783324755670416, -0.0047508389158269874]",1
1998,P1998,"[0.6915404361093128, 0.09694451341111456]",3


3. Visualization

Prepare the data into x and y components

In [19]:
pca_df[['x', 'y']] = pd.DataFrame(pca_df['pca_embeddings'].tolist(), index=pca_df.index)

In [20]:
def plot_clusters(pca_df, num_clusters=5):
    # Create the base cluster plot
    fig = px.scatter(
        pca_df,
        x='x',
        y='y',
        opacity=0.6,
        size_max=4,
        color= pca_df.cluster.astype(str),
        title='2D Visualization of Clusters with Recently Viewed Products',
        labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
        category_orders={'cluster': list(range(num_clusters))},
        # show the product id in the tooltip
        hover_data={'product_id': True}

    )

    # Update layout to add legend title and adjust plot settings
    fig.update_layout(legend_title_text='Clusters', legend=dict(x=1, y=1), width=600, height=500)

    return fig

fig = plot_clusters(pca_df)
fig.show()